# Data Visualisation
## Task 2.2.1 Correctness, Plot Design and Clarity
## Task 2.2.2 Interesting Point Annotation

This notebook loads detected violation records from MongoDB and produces interactive analytical
visualisations that support operational traffic monitoring and enforcement decisions.

**Visualisations produced:**
1. Daily Violation Count over Time - split by INSTANTANEOUS vs AVERAGE 
2. Rolling Average Violation Speed over Time - with spike/drop annotations 
3. Camera-level Violation Breakdown - which camera generates the most violations 
4. Hourly Heatmap - hour-of-day × violation-type pattern 

**Data source:** MongoDB `traffic_monitoring.violations` collection (populated by the streaming application).

## Step 1 Imports

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "plotly", "--quiet"])

In [ ]:
import pandas as pd
import numpy as np
import math
import warnings
warnings.filterwarnings("ignore")

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from pymongo import MongoClient

import plotly.io as pio
pio.renderers.default = "notebook"

HOST_IP = "host.docker.internal"
print("Imports complete.")

## Step 2 Load Violation Data

Load violation records from the MongoDB `violations` collection populated by the Spark streaming application.

In [ ]:
def load_from_mongo():
    """Load violation records from MongoDB."""
    client = MongoClient(host=HOST_IP, port=27017, serverSelectionTimeoutMS=3000)
    client.server_info()
    col = client["traffic_monitoring"]["violations"]
    rows = []
    for doc in col.find():
        # Flatten nested violations array into individual rows
        for v in doc.get("violations", []):
            rows.append({
                "car_plate": doc["car_plate"],
                "date": doc["date"],
                "violation_type": v["violation_type"],
                "camera_id_start": v["camera_id_start"],
                "camera_id_end": v["camera_id_end"],
                "timestamp_start": v["timestamp_start"],
                "timestamp_end": v["timestamp_end"],
                "speed_reading": v["speed_reading"],
            })
    client.close()
    return pd.DataFrame(rows)

print("Connecting to MongoDB...")
df = load_from_mongo()
print(f"Loaded {len(df):,} violation records from MongoDB.")

## Step 3 Preprocessing

In [ ]:
# Parse timestamps & extract time features
df["timestamp_start"] = pd.to_datetime(df["timestamp_start"])
df["date"]            = pd.to_datetime(df["date"])
df["day"]             = df["timestamp_start"].dt.floor("D")
df["hour_of_day"]     = df["timestamp_start"].dt.hour
df["day_of_week"]     = df["timestamp_start"].dt.day_name()

# Daily aggregations
daily_counts = (
    df.groupby(["day", "violation_type"])
    .size()
    .reset_index(name="count")
)

daily_speed = (
    df.groupby(["day", "violation_type"])["speed_reading"]
    .mean()
    .reset_index(name="avg_speed")
)

# Pivot for easier charting
daily_counts_pivot = daily_counts.pivot(index="day", columns="violation_type", values="count").fillna(0)
daily_speed_pivot  = daily_speed.pivot(index="day",  columns="violation_type", values="avg_speed").ffill()

print(f"Date range : {df['day'].min().date()}  →  {df['day'].max().date()}")
print(f"Total violations  : {len(df):,}")
print(f"Violation types   : {df['violation_type'].value_counts().to_dict()}")
print(f"Cameras involved  : {sorted(df['camera_id_start'].unique())}")

## Task 2.2.1 Plot 1: Daily Violation Count over Time

This chart shows the number of violations detected each day, split by violation type.

**Operational value:** Enforcement teams can identify which days or periods see peak violations and
allocate patrol resources accordingly. A sudden spike may indicate unusual traffic behaviour (holidays,
events), while a drop may indicate camera downtime.

In [ ]:
def get_annotations(series, label_prefix, color):
    """Identify notable points: global max/min, z-score spikes/drops, and P90 outliers."""
    annotations = []
    rolling = series.rolling(7, min_periods=1).mean()
    rolling_std = series.rolling(7, min_periods=1).std().fillna(0)

    # Global extremes
    idx_max = series.idxmax()
    idx_min = series.idxmin()
    annotations.append(dict(x=idx_max, y=series[idx_max], text=f"MAX: {series[idx_max]:.0f}",
                            color=color, symbol="triangle-up"))
    annotations.append(dict(x=idx_min, y=series[idx_min], text=f"MIN: {series[idx_min]:.0f}",
                            color=color, symbol="triangle-down"))

    # Z-score based anomaly detection (threshold: ±2 std from 7-day rolling mean)
    z = (series - rolling) / (rolling_std + 1e-9)
    for idx, val in series[z > 2.0].items():
        if idx not in (idx_max, idx_min):
            annotations.append(dict(x=idx, y=val, text=f"SPIKE: {val:.0f}",
                                    color="#f03e3e", symbol="star"))
    for idx, val in series[z < -2.0].items():
        if idx not in (idx_max, idx_min):
            annotations.append(dict(x=idx, y=val, text=f"DROP: {val:.0f}",
                                    color="#868e96", symbol="star"))

    # Mark days exceeding the 90th percentile
    p90 = series.quantile(0.90)
    for idx, val in series[series >= p90].items():
        if not any(a["x"] == idx for a in annotations):
            annotations.append(dict(x=idx, y=val, text=f"P90: {val:.0f}",
                                    color="#7950f2", symbol="diamond"))
    return annotations


COLORS = {"INSTANTANEOUS": "#3b5bdb", "AVERAGE": "#e67700"}
fig1 = go.Figure()

# 90th percentile reference lines
for vtype in daily_counts_pivot.columns:
    p90 = daily_counts_pivot[vtype].quantile(0.90)
    fig1.add_hline(y=p90, line_dash="dot", line_color=COLORS.get(vtype, "#555"),
                   opacity=0.5, annotation_text=f"{vtype} 90th pct: {p90:.0f}",
                   annotation_position="top right")

for vtype in daily_counts_pivot.columns:
    series = daily_counts_pivot[vtype]
    roll = series.rolling(7, min_periods=1).mean()
    color = COLORS.get(vtype, "#555")

    # Daily bars and 7-day rolling average
    fig1.add_trace(go.Bar(x=series.index, y=series.values,
                          name=f"{vtype} (daily)", marker_color=color,
                          opacity=0.25, showlegend=True))
    fig1.add_trace(go.Scatter(x=roll.index, y=roll.values, mode="lines",
                              name=f"{vtype} (7-day avg)",
                              line=dict(color=color, width=2.5)))

    # Annotations for notable points
    for a in get_annotations(series, vtype, color):
        fig1.add_annotation(
            x=a["x"], y=a["y"], text=f"<b>{a['text']}</b>",
            showarrow=True, arrowhead=2, arrowsize=1, arrowwidth=1.5,
            arrowcolor=a["color"], font=dict(size=10, color=a["color"]),
            bgcolor="rgba(255,255,255,0.85)",
            bordercolor=a["color"], borderwidth=1, ax=0, ay=-36
        )

fig1.update_layout(
    title=dict(text="Daily Violation Count over Time (INSTANTANEOUS vs AVERAGE)",
               font=dict(size=15)),
    xaxis_title="Date", yaxis_title="Number of Violations",
    legend_title="Series", barmode="overlay",
    hovermode="x unified", template="plotly_white", height=480
)

# Shade periods where violation count exceeds P90
for vtype in daily_counts_pivot.columns:
    p90 = daily_counts_pivot[vtype].quantile(0.90)
    series = daily_counts_pivot[vtype]
    color = COLORS.get(vtype, "#555")
    in_region = False
    start_date = None
    for date, is_above in (series >= p90).items():
        if is_above and not in_region:
            start_date = date
            in_region = True
        elif not is_above and in_region:
            fig1.add_vrect(x0=start_date, x1=date,
                           fillcolor=color, opacity=0.08,
                           layer="below", line_width=0)
            in_region = False
    if in_region:
        fig1.add_vrect(x0=start_date, x1=series.index[-1],
                       fillcolor=color, opacity=0.08,
                       layer="below", line_width=0)

fig1.show()

## Task 2.2.1 Plot 2: Average Violation Speed over Time

This chart shows how the average speed of detected violations changes day by day.

**Operational value:** A rising trend in average speed indicates drivers are becoming
more aggressive over time. Spikes above a threshold may warrant increased enforcement or
signage changes at the relevant camera location.

In [ ]:
# Top row: avg violation speed; bottom row: violation count for context
fig2 = make_subplots(rows=2, cols=1,
                     shared_xaxes=True,
                     subplot_titles=["Average Violation Speed (km/h)", "Daily Violation Count"],
                     row_heights=[0.65, 0.35],
                     vertical_spacing=0.08)

# Legal speed limits per violation type (used as reference lines)
SPEED_LIMITS = {"INSTANTANEOUS": 110, "AVERAGE": 90}

for vtype in daily_speed_pivot.columns:
    series = daily_speed_pivot[vtype].dropna()
    roll = series.rolling(7, min_periods=1).mean()
    color = COLORS.get(vtype, "#555")

    # 7-day rolling average gives a clearer trend than raw daily noise
    fig2.add_trace(go.Scatter(
        x=roll.index, y=roll.values,
        mode="lines", name=f"{vtype} speed (7-day avg)",
        line=dict(color=color, width=2.5)
    ), row=1, col=1)

    # Raw daily dots kept faint so they don't overwhelm the rolling line
    fig2.add_trace(go.Scatter(
        x=series.index, y=series.values,
        mode="markers", name=f"{vtype} speed (daily)",
        marker=dict(color=color, size=4, opacity=0.3),
        showlegend=True
    ), row=1, col=1)

    # Dashed line shows how far above the legal limit violations sit on average
    limit = SPEED_LIMITS.get(vtype, 110)
    fig2.add_hline(y=limit, line_dash="dash", line_color=color,
                   opacity=0.5,
                   annotation_text=f"{vtype} limit: {limit} km/h",
                   annotation_position="bottom right",
                   row=1, col=1)

    # Annotate the single highest and lowest speed days
    idx_max = series.idxmax()
    idx_min = series.idxmin()
    for idx, val, label in [(idx_max, series[idx_max], f"PEAK {series[idx_max]:.1f} km/h"),
                             (idx_min, series[idx_min], f"MIN {series[idx_min]:.1f} km/h")]:
        fig2.add_annotation(
            x=idx, y=val, text=f"<b>{label}</b>",
            showarrow=True, arrowhead=2, arrowcolor=color,
            font=dict(size=9, color=color),
            bgcolor="rgba(255,255,255,0.85)", bordercolor=color, borderwidth=1,
            ax=0, ay=-32, row=1, col=1
        )

    # Flag days where speed jumps more than 2.5 std above the rolling mean
    roll_std = series.rolling(7, min_periods=1).std().fillna(0)
    z = (series - roll) / (roll_std + 1e-9)
    for idx, val in series[z > 2.5].items():
        if idx not in (idx_max, idx_min):
            fig2.add_annotation(
                x=idx, y=val, text=f"<b>SPIKE {val:.1f}</b>",
                showarrow=True, arrowhead=2, arrowcolor="#f03e3e",
                font=dict(size=9, color="#f03e3e"),
                bgcolor="rgba(255,240,240,0.9)", bordercolor="#f03e3e", borderwidth=1,
                ax=0, ay=-28, row=1, col=1
            )

    # Mini count line in lower subplot — helps correlate speed spikes with volume changes
    if vtype in daily_counts_pivot.columns:
        fig2.add_trace(go.Scatter(
            x=daily_counts_pivot.index,
            y=daily_counts_pivot[vtype].rolling(7, min_periods=1).mean(),
            mode="lines", name=f"{vtype} count (7-day avg)",
            line=dict(color=color, width=1.5),
            showlegend=False
        ), row=2, col=1)

fig2.update_layout(
    title=dict(text="Violation Speed Pattern over Time with Anomaly Annotations",
               font=dict(size=15)),
    yaxis_title="Speed (km/h)",
    yaxis2_title="Count",
    hovermode="x unified",
    template="plotly_white",
    height=580,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)
fig2.update_xaxes(title_text="Date", row=2, col=1)
fig2.show()

## Task 2.2.1 Plot 3: Camera-level Violation Breakdown

This chart breaks down the total number of violations by camera (start camera) and violation type.

**Operational value:** Identifies which camera checkpoint is the highest-risk location.
Camera 3 has a lower speed limit (90 km/h) and typically records more violations because
drivers carry speed from the earlier segment. This justifies prioritising enforcement resources
near Camera 3.

In [ ]:
# Aggregate by camera and violation type
cam_counts = (
    df.groupby(["camera_id_start", "violation_type"])
    .size()
    .reset_index(name="count")
)
cam_counts["camera_label"] = "Camera " + cam_counts["camera_id_start"].astype(str)

fig3 = px.bar(
    cam_counts,
    x="camera_label", y="count",
    color="violation_type",
    barmode="group",
    text="count",
    color_discrete_map={"INSTANTANEOUS": "#3b5bdb", "AVERAGE": "#e67700"},
    labels={"camera_label": "Camera", "count": "Number of Violations",
            "violation_type": "Violation Type"},
    title="Violation Count by Camera and Type"
)

# Mark the single camera+type combo with the most violations
max_cam = cam_counts.loc[cam_counts["count"].idxmax()]
fig3.add_annotation(
    x=max_cam["camera_label"],
    y=max_cam["count"],
    text=f"<b>HIGHEST: {max_cam['count']:,}<br>{max_cam['violation_type']}</b>",
    showarrow=True, arrowhead=2, arrowcolor="#e03131",
    font=dict(size=10, color="#e03131"),
    bgcolor="rgba(255,240,240,0.9)", bordercolor="#e03131", borderwidth=1,
    ax=40, ay=-40
)

# Show counts above bars; hide if bars are too narrow to fit the number
fig3.update_traces(texttemplate="%{text:,}", textposition="outside")
fig3.update_layout(
    template="plotly_white",
    height=420,
    uniformtext_minsize=9,
    uniformtext_mode="hide",
    legend_title="Violation Type"
)
fig3.show()

## Task 2.2.2 Plot 4: Violation Heatmap by Hour of Day

This heatmap shows the distribution of violations by hour of day and violation type.

**Operational value:** Reveals the time-of-day pattern of speeding behaviour.
Peak hours (e.g. morning rush 07:00–09:00, evening 17:00–19:00) can guide dynamic enforcement
scheduling. An unexpectedly high late-night count may indicate issues with the camera's
false-positive rate or aggressive night-time driving.

In [ ]:
hourly = (
    df.groupby(["hour_of_day", "violation_type"])
    .size()
    .reset_index(name="count")
)

fig4 = px.density_heatmap(
    hourly,
    x="hour_of_day", y="violation_type",
    z="count",
    color_continuous_scale="Blues",
    labels={"hour_of_day": "Hour of Day (24h)", "violation_type": "Violation Type",
            "count": "Violation Count"},
    title="Violation Count Heatmap — Hour of Day × Violation Type",
    text_auto=True  # print the count inside each cell
)

# Find the hour+type cell with the highest count and call it out
peak_row = hourly.loc[hourly["count"].idxmax()]
fig4.add_annotation(
    x=peak_row["hour_of_day"],
    y=peak_row["violation_type"],
    text=f"<b>PEAK HOUR<br>{int(peak_row['hour_of_day']):02d}:00</b>",
    showarrow=True, arrowhead=2, arrowcolor="#e03131",
    font=dict(size=11, color="#e03131"),
    bgcolor="rgba(255,255,255,0.9)", bordercolor="#e03131", borderwidth=1.5,
    ax=50, ay=-30
)

# dtick=1 forces every hour to show on the x-axis instead of auto-skipping
fig4.update_layout(
    template="plotly_white",
    height=360,
    xaxis=dict(dtick=1, title="Hour of Day (24h)"),
    coloraxis_colorbar=dict(title="Count")
)
fig4.show()

## Summary Statistics

In [ ]:
summary = df.groupby("violation_type").agg(
    total_violations=("car_plate", "count"),
    unique_vehicles=("car_plate", "nunique"),
    avg_speed_kmh=("speed_reading", "mean"),
    max_speed_kmh=("speed_reading", "max"),
    min_speed_kmh=("speed_reading", "min"),
).round(2)

print("=" * 60)
print("VIOLATION SUMMARY")
print("=" * 60)
print(summary.to_string())
print()
print(f"Total violations : {len(df):,}")
print(f"Date range       : {df['day'].min().date()} -> {df['day'].max().date()}")

# Daily counts already computed in preprocessing, so just find the max row
busiest = daily_counts.loc[daily_counts["count"].idxmax()]
print(f"Busiest day      : {busiest['day'].date()} "
      f"({int(busiest['count']):,} {busiest['violation_type']} violations)")

## Task 2.2.2 Interesting Point Annotation: Operational Insights

The visualisations above annotate the following categories of notable points:

### Annotation Types and Their Operational Meaning

| Annotation | Definition | Operational Significance |
|------------|-----------|--------------------------|
| **MAX** | Day with the globally highest violation count or speed | Indicates peak enforcement demand — may correlate with public holidays, weekends, or major events. Resources should be pre-positioned on such days. |
| **MIN** | Day with the globally lowest violation count or speed | May indicate camera downtime, adverse weather reducing traffic, or a reporting gap. Should be cross-referenced against maintenance logs. |
| **SPIKE** | Day where count or speed is >2 standard deviations above the 7-day rolling mean | Sudden surges that are statistically anomalous. Could indicate a race, reckless driving cluster, or a newly popular route. Warrants immediate investigation. |
| **DROP** | Day where count or speed is >2 standard deviations below the rolling mean | Sudden suppressions. Could indicate equipment failure or an unusual traffic event (road closure). |
| **PEAK HOUR** | Hour of day with the highest total violation count in the heatmap | Defines the optimal window for mobile enforcement deployment or dynamic speed limit display activation. |
| **HIGHEST CAMERA** | Camera location recording the most violations | Identifies the single most critical enforcement point. Camera 3 (speed limit 90 km/h) typically dominates due to drivers carrying speed from the A→B segment. |

### Why These Visualisations Matter Operationally

**Plot 1 Daily Count Trend:**  
The dual INSTANTANEOUS / AVERAGE series allows enforcement managers to distinguish between
*point-in-time* speeding (a driver momentarily exceeding the limit) and *sustained* speeding
(a driver covering an entire segment at excessive speed). AVERAGE violations are generally more
dangerous because they indicate deliberate, prolonged speeding rather than a brief lapse.

**Plot 2 Speed Pattern with Anomalies:**  
Tracking the *average speed* of violations over time reveals whether driving behaviour is
deteriorating. A rising speed trend should trigger regulatory review. The speed-limit reference
lines provide immediate visual context for how far above the legal threshold detected violations sit.

**Plot 3 Camera Breakdown:**  
Camera-level aggregation reveals spatial hot-spots. If Camera 3 consistently records 3× more
violations than Cameras 1 and 2, it justifies a permanent enforcement presence or infrastructure
change (better signage, rumble strips) at that location.

**Plot 4 Hourly Heatmap:**  
Time-of-day patterns enable *predictive* enforcement scheduling. If violations peak between
07:00 and 09:00, mobile patrol units can be deployed proactively during the morning commute
rather than reactively after incidents occur.